## Visualize canon threads

Group the indexed `result/sample-data.json` hits by `thread_id` / `canon_order`.
`_id` suffix encodes dedup: no suffix = original, `m` = near-duplicate (Levenshtein > 0.9),
`mm` = exact `content_hash` duplicate. Each duplicate hangs off the slot it collapsed into.


In [ ]:
!pip install requests

In [ ]:
import re
from collections import defaultdict
import requests

resp = requests.post(
    "http://my.local/es/email/_search",
    json={"size": 1000, "sort": [{"canon_order": "asc"}]},
)
resp.raise_for_status()
hits = [h["_source"] for h in resp.json()["hits"]["hits"]]


def variant(es_id):
    m = re.search(r"m+$", es_id)
    return m.group(0) if m else ""


threads = defaultdict(lambda: defaultdict(list))
for s in hits:
    threads[s["thread_id"]][s["canon_order"]].append(s)

for tid, slots in threads.items():
    subj = next(iter(slots.values()))[0]["subject"]
    print(f"THREAD {tid[:8]}  {subj}")
    for order in sorted(slots):
        rows = sorted(slots[order], key=lambda s: len(variant(s["id"])))
        for i, s in enumerate(rows):
            v = variant(s["id"])
            tag = "orig" if not v else ("=dup" if len(v) > 1 else "~dup")
            branch = f"  [{order}]" if i == 0 else "       +"
            who = ",".join(s["from"])
            print(f"{branch} {tag:4} {s['id']:11} {who:18} {s['content']}")
    print()